# Set up

Retrieve the data we'll use from the git repository: a bunch of markdown files

In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)


# Q1. How many lesson pages

In [2]:
print(len(documents))

72


# Q2. Indexing and Searching

Let's use minsearch to index and search the documents. `index` stores the indexed documents, which have two fields: `filename` and `content`

In [3]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

The indexed documents are ready!! Let's run our first search

In [4]:
question = "How does the agentic loop keep calling the model until it stops?"

search_results = index.search(
    question,
    num_results=5
)

The first result is

In [5]:
search_results[0]["filename"]

'01-agentic-rag/lessons/14-agentic-loop.md'

# Q3. RAG Implementation

Create a the `RAGBase` class that will use the `minsearch` index from the previous questions to search for relevant documents and build a prompt. Then it will use the client passed as an argument to query the LLM and return the response.

Initialize the OpenAI client:

In [6]:
INSTRUCTIONS = '''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
'''

PROMPT_TEMPLATE = '''
QUESTION: {question}

CONTEXT:
{context}
'''.strip()


class RAGBase:

    def __init__(
        self,
        index,
        llm_client,
        instructions=INSTRUCTIONS,
        prompt_template=PROMPT_TEMPLATE,
        course='llm-zoomcamp',
        model='gpt-5.4-mini'
    ):
        self.index = index
        self.llm_client = llm_client
        self.instructions = instructions
        self.course = course
        self.prompt_template = prompt_template
        self.model = model

    def search(self, query, num_results=5):
        boost_dict = {'content': 3.0, 'filename': 0.5}
        filter_dict = {}

        return self.index.search(
            query,
            num_results=num_results,
            boost_dict=boost_dict,
            filter_dict=filter_dict
        )

    def build_context(self, search_results):
        lines = []

        for doc in search_results:
            lines.append(doc['filename'])
            lines.append('content: ' + doc['content'])
            lines.append('')

        return '\n'.join(lines).strip()

    def build_prompt(self, query, search_results):
        context = self.build_context(search_results)
        return self.prompt_template.format(
            question=query, context=context
        )

    def llm(self, prompt):
        input_messages = [
            {'role': 'developer', 'content': self.instructions},
            {'role': 'user', 'content': prompt}
        ]

        response = self.llm_client.responses.create(
            model=self.model,
            input=input_messages
        )

        return response

    def rag(self, query):
        search_results = self.search(query)
        prompt = self.build_prompt(query, search_results)
        response = self.llm(prompt)
        return response.output_text, response.usage.input_tokens

In [7]:
from openai import OpenAI
import os

from dotenv import load_dotenv

load_dotenv('../.env')

openai_client = OpenAI()


In [8]:
from pprint import pprint
from IPython.display import display, Markdown

rag = RAGBase(index,openai_client)
# pprint(rag.llm('who are you?'))
answer,input_tokens = rag.rag('How does the agentic loop keep calling the model until it stops?')
display(Markdown(f"We have consummed `{input_tokens}` input tokens"))
display(Markdown(f"**This is the answer that we got:**"))
display(Markdown(answer))

We have consummed `7121` input tokens

**This is the answer that we got:**

It keeps calling the model in a `while True` loop.

Each turn it:
1. sends the current `messages` to the model,
2. checks the response for any `function_call`s,
3. runs those tools and appends the outputs to `messages`,
4. repeats.

It stops when a response comes back with no function calls. The code uses a `has_function_calls` flag, and if that stays `False`, it breaks out of the loop.

# Q4. Chunking

In [9]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
display(Markdown(f"Chunks: `{len(chunks)}`"))

Chunks: `295`

# Q5. RAG with chunking

In [10]:
chunked_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

chunked_index.fit(chunks)

rag = RAGBase(chunked_index,openai_client)
# pprint(rag.llm('who are you?'))
answer,chunked_input_tokens = rag.rag('How does the agentic loop keep calling the model until it stops?')
display(Markdown(f"We have consummed `{chunked_input_tokens}` input tokens, **which is about {round(input_tokens/chunked_input_tokens,2)} times less**"))
display(Markdown(f"**This is the answer that we got:**"))
display(Markdown(answer))

We have consummed `2304` input tokens, **which is about 3.09 times less**

**This is the answer that we got:**

It keeps calling the model in a `while True` loop and checks each response for any `function_call` items.

- If the model returns a function call, the code runs the tool, appends the result to `messages`, and keeps looping.
- If the model returns a normal `message` with no function calls, `has_function_calls` stays `False`, and the loop breaks.

So the stop condition is: **no function calls in the current turn**.